In [375]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.animation import FuncAnimation
from IPython.display import Image
import numba as nb
from numba import njit
from scipy.spatial import cKDTree

plt.rcParams['figure.figsize'] = [10, 10] #Figure size
plt.rcParams["axes.titlesize"] = 20   # Title size
plt.rcParams["axes.labelsize"] = 15   # X and Y label size
plt.rcParams["legend.fontsize"] = 13     # Legend size
plt.rcParams["xtick.labelsize"] = 15     # X-axis tick labels
plt.rcParams["ytick.labelsize"] = 15     # Y-axis tick labels
import matplotlib
matplotlib.use("TkAgg")
np.random.seed(42)


In [357]:

def intvals(species,min = -1, max = 1,no_selfint = True,symmetric = True):
    # Interaction matrix for different species, symmetric
    interaction = np.random.uniform(min,max, size = (species,species))
    if symmetric:
        intmat = (interaction+interaction.T)/2
    else:
        intmat = interaction
    if no_selfint:
        intmat = (np.ones((species,species))-np.eye(species,k=0))*intmat
    return intmat


def speciesindex(N,species):
    #Assigns a random species to each particle
    return np.random.choice(np.arange(0,species),size = (N))
def interaction(interactionmatrix,dx,dy,speciesindex,r_cut = 1,expo=1):
    
    N = len(speciesindex)
    distance = dx**2+dy**2
    r = np.sqrt(distance)
    mask = (r < r_cut) & (r > 0)
    Fx = np.zeros_like(dx)
    Fy = np.zeros_like(dy)
    pair_interactions = interactionmatrix[speciesindex[:, np.newaxis], speciesindex[np.newaxis, :]]
    factor = (pair_interactions[mask]/((r[mask]**(2)+1e-4)))#*np.exp(expo*r[mask])))
    Fx[mask] = -factor * dx[mask]
    Fy[mask] = -factor * dy[mask]
    return Fx,Fy


In [ ]:
def update():
    global positions, velocities,densitylist,N,L,dt,ints,spcs,rmin,expo
    dx = positions[:, 0][:, np.newaxis] - positions[:, 0][np.newaxis, :]
    dy = positions[:, 1][:, np.newaxis] - positions[:, 1][np.newaxis, :]
    dx = dx - L * np.round(dx / L)
    dy = dy - L * np.round(dy / L)
    Fx,Fy =interaction(ints,dx,dy,spcs,rmin)
    Fx_net = np.sum(Fx, axis=1)
    Fy_net = np.sum(Fy, axis=1)
    new_velx = dt*Fx_net
    new_vely = dt*Fy_net
    velocities += dt*np.stack((new_velx,new_vely),axis = 1)-0.01*velocities
    positions  = (positions + dt*velocities)%L 


In [314]:
def gif():
    global positions, velocities, T,species
    fig, ax = plt.subplots(figsize=(6, 6))
    scat = ax.scatter(positions[:,0], positions[:,1], c=spcs, cmap='tab10',vmin=0,vmax=species-1
)

    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_title("Particle species simulation")

    def animate(frame):
        scat.set_offsets(positions)
        #scat.set_UVC(velocities[:, 0], velocities[:, 1])
        update()
        return scat,

    ani = FuncAnimation(fig, animate, frames=T, interval=30)
    plt.close()

    writergif = animation.PillowWriter(fps=30)
    ani.save('test.gif',writer=writergif)
    Image(open('test.gif','rb').read())

# Run simulation - Visual

In [374]:

species = 4
N = 400
L = 1
dt = 0.01
spcs = speciesindex(N,species)
ints = intvals(species,symmetric = False,no_selfint=True)
print(ints)
velocities = np.zeros((N,2))
positions = np.random.uniform(0,L,size = (N,2))
rmin = 1
expo = 200
def run_simulation():
    global positions, velocities

    fig, ax = plt.subplots(figsize=(6, 6))

    scat = ax.scatter(
        positions[:, 0],
        positions[:, 1],
        c=spcs,
        cmap="tab10",
        vmin=0,
        vmax=species - 1
    )

    ax.set_xlim(0, L)
    ax.set_ylim(0, L)
    ax.set_title("Particle species simulation")

    def animate(frame):
        update()
        scat.set_offsets(positions)
        return scat,

    ani = FuncAnimation(fig, animate, interval=30, cache_frame_data=False)    
    plt.show()
    return ani  
anim = run_simulation()

[[-0.         -0.05900372  0.00970173  0.18199972]
 [-0.05746793 -0.          0.27528435  0.45211762]
 [-0.18060471  0.64615955 -0.          0.79260577]
 [-0.61791408  0.16812011  0.83830587 -0.        ]]


In [306]:
species = 1
N = 100
L = 1
dt = 0.005
spcs = speciesindex(N,species)
ints = intvals(species,max = -0.9,no_selfint=False)
velocities = np.zeros((N,2))
positions = np.random.uniform(0,L,size = (N,2))
t = 0
print(ints)

def conc(positions,rmin,L = 1):
    tree = cKDTree(positions)
    counts = tree.query_ball_point(positions, rmin, return_length=True)
    counts = counts - 1
    area = np.pi * rmin**2
    local_concentration = counts / (area)
    return local_concentration
    
rmin = 0.1
orderparam_vel = []
orderparam_conc = []
timelike = []

while t < 10:
    t += dt
    update()
    timelike.append(t)
    #orderparam_vel.append(np.mean(velocities)/np.sum(abs(velocities)))
    local_conc = conc(positions,rmin)
    orderparam_conc.append(L**2*np.mean(local_conc)/N) #Multiplied by L**2/N to make it unitless. 
plt.plot(timelike,orderparam_conc)
plt.show()

[[-0.96293507]]


In [260]:
plt.semilogy(timelike,orderparam_conc)
plt.show()